# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện
**Track 4 · VinUniversity AICB 2026**

Notebook hoàn chỉnh các phần từ Part 0 đến Part 4.
Có thể chạy trên Google Colab (GPU T4) hoặc local GPU/CPU.


In [ ]:
# ===== [GOOGLE COLAB SETUP] Chạy ô này đầu tiên nếu bạn đang dùng Colab =====
import os, sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/qlam1208/K4-Track4-Day1-Neuralnetwork.git'
REPO_NAME = 'K4-Track4-Day1-Neuralnetwork'

if IN_COLAB:
    print("--> Đang chạy trên Google Colab. Bắt đầu thiết lập môi trường...")
    if not os.path.exists(REPO_NAME):
        print(f"--> Đang clone repo từ {REPO_URL}...")
        !git clone {REPO_URL}
    else:
        print("--> Repo đã tồn tại, đang pull cập nhật mới nhất...")
        !cd {REPO_NAME} && git pull
    
    # Chuyển thư mục làm việc vào thư mục code của repo
    target_dir = f"/content/{REPO_NAME}/code"
    if os.path.exists(target_dir):
        os.chdir(target_dir)
        print(f"--> Đã chuyển thư mục làm việc về: {os.getcwd()}")
    
    # Cài đặt thư viện cần thiết nếu thiếu
    !pip install -q openpyxl
    print("--> Thiết lập môi trường Colab hoàn tất!")
else:
    print(f"--> Đang chạy ở máy local. Thư mục hiện tại: {os.getcwd()}")


In [ ]:
# ===== 1. Cấu hình môi trường và đường dẫn =====
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Thiết lập đường dẫn tương đối để chạy được cả ở local và Colab
REPO_ROOT = ".." if Path("../scripts").exists() else "."
OUT_DIR = ".." if Path("../figures").exists() or Path("../results").exists() else "."

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

# Chọn thiết bị tính toán
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Sử dụng GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Sử dụng Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Sử dụng CPU")

print(f"PyTorch version: {torch.__version__}")

# Thêm thư mục code vào sys.path để import các module
if str(Path(".").resolve()) not in sys.path:
    sys.path.append(str(Path(".").resolve()))

from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


## Part 0 — Chuẩn bị dữ liệu
1. Chia dữ liệu metadata thành `train.npz` và `eval.npz` (nếu chưa có).
2. Tách 20% validation phân tầng từ train (seed 42).
3. Chuẩn hóa 10 cột liên tục bằng mean/std của train-sub. Đưa toàn bộ lên device.


In [ ]:
# 1. Chạy split_data.py nếu chưa có train.npz
processed_dir = Path(f"{REPO_ROOT}/data/processed")
if not (processed_dir / "train.npz").exists() or not (processed_dir / "eval.npz").exists():
    print("Chưa có data/processed. Đang chạy split_data.py...")
    split_cmd = [
        sys.executable,
        f"{REPO_ROOT}/scripts/split_data.py",
        "--data", f"{REPO_ROOT}/data/covtype.csv.gz",
        "--meta", f"{REPO_ROOT}/data/split_metadata.csv",
        "--out", str(processed_dir),
    ]
    subprocess.run(split_cmd, check=True)

# 2. Nạp dữ liệu và chuẩn hóa
data = prepare_data(device=device, val_fraction=0.2, seed=42, processed_dir=str(processed_dir))

# Kiểm tra mean và std của 10 cột đầu trên train
X_tr_num = data["X_tr"][:, :10]
means = X_tr_num.mean(dim=0).cpu().numpy()
stds = X_tr_num.std(dim=0).cpu().numpy()
print("\nKiểm tra 10 cột số sau chuẩn hóa trên X_tr:")
print("  Mean xấp xỉ 0:", np.allclose(means, 0, atol=1e-2))
print("  Std xấp xỉ 1: ", np.allclose(stds, 1, atol=1e-2))


## Part 1 — Định nghĩa Model & Kiểm tra "Sức khỏe" ban đầu
- Kiến trúc `M-base`: `54 → 256 → 128 → 7` (47 879 tham số).
- Phép thử:
  1. Kiểm tra shape logits `(B, 7)`.
  2. Loss bước 0 trước khi học $\approx \ln(7) \approx 1.946$.
  3. Overfit thành công một batch nhỏ 20 mẫu (loss $\to 0$).
  4. Kiểm tra gradient flow qua tất cả các lớp.


In [ ]:
# Phép thử 1 & 2: Tạo model, đếm tham số, kiểm tra shape
set_seed(42)
test_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(test_model)
print(f"Số tham số M-base: {n_params} (Kỳ vọng: {EXPECTED_PARAMS[(256, 128)]})")
assert n_params == EXPECTED_PARAMS[(256, 128)], "Số tham số không khớp!"

dummy_x = torch.randn(8, 54, device=device)
dummy_logits = test_model(dummy_x)
print(f"Output shape cho batch 8: {dummy_logits.shape} (Kỳ vọng: (8, 7))")
assert dummy_logits.shape == (8, 7), "Output shape không khớp!"

# Phép thử 3: Loss bước 0 trên tập val
step0_eval = evaluate(test_model, data["X_val"], data["y_val"], loss_name="ce")
print(f"Loss bước 0 trên val: {step0_eval['loss']:.4f} (Kỳ vọng ln(7) = {np.log(7):.4f})")

# Phép thử 4: Overfit 20 mẫu
print("\nĐang kiểm tra khả năng overfit trên 20 mẫu...")
x_tiny, y_tiny = data["X_tr"][:20], data["y_tr"][:20]
tiny_opt = torch.optim.Adam(test_model.parameters(), lr=0.01)

for step in range(200):
    tiny_opt.zero_grad()
    loss = F.cross_entropy(test_model(x_tiny), y_tiny)
    loss.backward()
    tiny_opt.step()

final_tiny_loss = loss.item()
print(f"Loss sau 200 bước trên 20 mẫu: {final_tiny_loss:.6f}")
assert final_tiny_loss < 0.05, "Model không thể overfit 20 mẫu!"
print("=> Model qua được tất cả các bài kiểm tra sức khỏe ban đầu!")


## Part 2 — Pipeline Huấn luyện & Baseline
Baseline cố định theo yêu cầu:
- Mạng `M-base` (`54 → 256 → 128 → 7`), ReLU, He init, FP32
- Optimizer: SGD + Momentum 0.9, batch size 512, 20 epochs
- Chạy 3 seed (1, 2, 3) để đo độ nhiễu giữa các lần chạy và xác định ngưỡng $2\sigma$.


In [ ]:
# Chạy baseline với 3 seed khác nhau
baseline_results = []
baseline_seeds = [1, 2, 3]
base_lr = 0.05

for s in baseline_seeds:
    exp_id = f"base-s{s}"
    print(f"\n--- Đang chạy {exp_id} ---")
    cfg = {
        **DEFAULT_CFG,
        "exp_id": exp_id,
        "group": "baseline",
        "description": f"Baseline M-base (Seed {s})",
        "lr": base_lr,
        "seed": s,
        "epochs": 20,
    }
    res = run_experiment(cfg, data)
    baseline_results.append(res)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{exp_id}.png")
    print(f"Done {exp_id}: Best val loss = {res['summary']['best_val_loss']:.4f}, Macro-F1 = {res['summary']['val_macro_f1']:.4f}")

# Đo độ nhiễu giữa các seed
f1_scores = [r["summary"]["val_macro_f1"] for r in baseline_results]
f1_mean = np.mean(f1_scores)
f1_std = np.std(f1_scores)
noise_threshold = 2 * f1_std

print(f"\n[Baseline Summary qua {len(baseline_seeds)} seeds]")
print(f"  Val Macro-F1: {f1_mean:.4f} ± {f1_std:.4f}")
print(f"  Ngưỡng nhiễu 2σ: {noise_threshold:.4f}")


## Part 3 — Hệ thống Thí nghiệm (Experiments)
Khảo sát các yếu tố ảnh hưởng tới huấn luyện (mỗi lần chỉ đổi một yếu tố so với baseline):
1. **Hàm mất mát**: CE vs MSE.
2. **Bộ tối ưu hoá**: SGD vs SGD+Momentum vs Adam vs AdamW.
3. **Tốc độ học (Learning Rate)**: 0.1, 0.01, 0.001.
4. **Dropout**: 0.0, 0.2, 0.5.
5. **Cắt gradient (Clipping)**: Thử nghiệm ổn định gradient khi lr lớn.
6. **Khởi tạo tham số**: Zeros, Normal, Xavier, He.


In [ ]:
# Danh sách cấu hình thí nghiệm
all_experiments = []

# Chủ đề 1: Loss (MSE)
all_experiments.append({
    **DEFAULT_CFG,
    "exp_id": "loss-mse",
    "group": "loss",
    "description": "MSE Loss thay cho CrossEntropy",
    "loss": "mse",
    "lr": 0.01,
})

# Chủ đề 2: Optimizer
for opt in ["sgd", "adam", "adamw"]:
    all_experiments.append({
        **DEFAULT_CFG,
        "exp_id": f"opt-{opt}",
        "group": "optimizer",
        "description": f"Bộ tối ưu {opt}",
        "optimizer": opt,
        "lr": 0.001 if "adam" in opt else 0.05,
    })

# Chủ đề 3: Learning Rate
for lr in [0.01, 0.1]:
    all_experiments.append({
        **DEFAULT_CFG,
        "exp_id": f"lr-{lr}",
        "group": "hparam",
        "description": f"Khảo sát Learning Rate = {lr}",
        "lr": lr,
    })

# Chủ đề 4: Dropout
for drop in [0.2, 0.5]:
    all_experiments.append({
        **DEFAULT_CFG,
        "exp_id": f"drop-{drop}",
        "group": "dropout",
        "description": f"Dropout rate = {drop}",
        "dropout": drop,
    })

# Chủ đề 5: Gradient Clipping
all_experiments.append({
    **DEFAULT_CFG,
    "exp_id": "clip-1.0",
    "group": "clipping",
    "description": "Cắt gradient L2 norm max = 1.0",
    "clip_norm": 1.0,
})

# Chủ đề 6: Khởi tạo trọng số
for init_method in ["normal", "xavier", "zeros"]:
    all_experiments.append({
        **DEFAULT_CFG,
        "exp_id": f"init-{init_method}",
        "group": "init",
        "description": f"Khởi tạo trọng số {init_method}",
        "init": init_method,
    })

# Chạy tất cả thí nghiệm
exp_results = {}
for exp_cfg in all_experiments:
    eid = exp_cfg["exp_id"]
    print(f"\n--- Đang thực hiện thí nghiệm {eid} ({exp_cfg['group']}) ---")
    res = run_experiment(exp_cfg, data)
    exp_results[eid] = res
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{eid}.png")
    print(f"Hoàn thành {eid}: val_loss={res['summary']['best_val_loss']:.4f}, val_f1={res['summary']['val_macro_f1']:.4f}")


In [ ]:
# Vẽ các biểu đồ so sánh theo nhóm
groups = set(cfg["group"] for cfg in all_experiments)

for grp in groups:
    grp_results = [res for eid, res in exp_results.items() if res["cfg"]["group"] == grp]
    if baseline_results:
        grp_results = [baseline_results[0]] + grp_results
    plot_compare(grp_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_{grp}.png", title=f"So sánh nhóm {grp} (Val Macro-F1)")
print("Đã vẽ xong các biểu đồ so sánh nhóm!")


## Part 4 — Đánh giá cuối trên tập Eval & Xuất bảng Excel
1. Chọn cấu hình tối ưu nhất theo Validation Macro-F1.
2. Dự đoán trên toàn bộ `X_eval`, xuất `predictions_eval.csv`.
3. Chạy `scripts/evaluate.py` để tính điểm chính thức.
4. Xuất bảng `experiments.xlsx` từ danh sách các kết quả JSON.


In [ ]:
# 1. Tìm cấu hình tốt nhất dựa trên tập val
all_runs = baseline_results + list(exp_results.values())
valid_runs = [r for r in all_runs if not r["summary"]["diverged"]]
best_run = max(valid_runs, key=lambda r: r["summary"]["val_macro_f1"])

print(f"Cấu hình tốt nhất được chọn từ val: {best_run['cfg']['exp_id']}")
print(f"  Val Macro-F1: {best_run['summary']['val_macro_f1']:.4f}")
print(f"  Val Accuracy: {best_run['summary']['val_acc']:.4f}")

# 2. Suy luận trên tập eval và ghi file predictions_eval.csv
pred_path = f"{OUT_DIR}/predictions_eval.csv"
final_eval(best_run["cfg"], best_run, data, pred_path)

# 3. Chạy evaluate.py để chấm điểm chính thức
eval_out_json = f"{OUT_DIR}/eval_result.json"
eval_cmd = [
    sys.executable, f"{REPO_ROOT}/scripts/evaluate.py",
    "--pred", pred_path,
    "--data", f"{REPO_ROOT}/data/covtype.csv.gz",
    "--meta", f"{REPO_ROOT}/data/split_metadata.csv",
    "--out", eval_out_json,
]
subprocess.run(eval_cmd, check=True)

with open(eval_out_json, "r", encoding="utf-8") as f:
    eval_result = json.load(f)

print(f"\n=== KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC TRÊN EVAL ===")
print(f"  Macro-F1 : {eval_result['macro_f1']:.4f}")
print(f"  Accuracy : {eval_result['accuracy']:.4f}")


In [ ]:
# 4. Xuất toàn bộ kết quả vào experiments.xlsx
results_all = load_results(f"{OUT_DIR}/results")
rows = []

for r in results_all:
    eid = r["cfg"]["exp_id"]
    if eid == best_run["cfg"]["exp_id"]:
        # Điền eval score cho cấu hình tốt nhất
        row = to_row(r, eval_scores={"acc": eval_result["accuracy"], "macro_f1": eval_result["macro_f1"]}, notes="Cấu hình cuối cùng được chọn")
    else:
        row = to_row(r)
    rows.append(row)

template_xlsx = f"{REPO_ROOT}/templates/experiment_table_template.xlsx"
out_xlsx = f"{OUT_DIR}/experiments.xlsx"
write_xlsx(rows, template_xlsx, out_xlsx)
print(f"Đã xuất thành công bảng {out_xlsx} với {len(rows)} thí nghiệm!")
